# Direct air capture, end to end

trailrunner computes a life cycle inventory by traversing a supply chain of
Python *models* — one per technology — instead of solving a fixed matrix. A
model reads its parameters from a [trailpack](https://github.com/TimoDiepers/trailpack)
parquet file and answers one question: *given this demand, what did I produce,
what do I need, and what did I emit?*

`DirectAirCapture` is the worked example, and it exists because of one number:
the heat needed to regenerate the sorbent is not a constant. Colder, drier air
carries less CO2 and less water to the sorbent per unit of air moved, so heat
and fan work per kilogram captured go up. A coefficient in a table cannot say
that; a function can.

This notebook builds a parameter file, reads it, runs the model on a single
demand, and then lets the orchestrator walk outward from it. Everything runs on
trailrunner's own dependencies — no extras to install beyond `pyarrow`.

## 1. Parameters live in a parquet file, units and all

Model code holds the *behaviour*; the *numbers* come from a parquet file with a
Frictionless `datapackage.json` embedded in its schema metadata. That descriptor
is what makes the columns self-describing: a unit at `fields[].unit.name` and a
[PyST](https://vocab.sentier.dev) concept IRI at `fields[].rdfType`.

In practice trailpack writes these files. Here `write_parameters` writes small
ones by hand so the notebook is self-contained — one per process, as trailpack
would: first the four columns the DAC model asks for, over two locations and two
years.


In [1]:
import json
import tempfile
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq

workdir = Path(tempfile.mkdtemp())


def write_parameters(name, rows, fields):
    """Write rows to parquet with a Frictionless descriptor in the schema metadata."""
    path = workdir / name
    table = pa.Table.from_pylist(rows)
    datapackage = {
        "name": path.stem,
        "resources": [
            {
                "name": "parameters",
                "path": path.name,
                "fields": [
                    {
                        "name": field["name"],
                        "type": field["type"],
                        **({"unit": {"name": field["unit"]}} if "unit" in field else {}),
                        **({"rdfType": field["iri"]} if "iri" in field else {}),
                    }
                    for field in fields
                ],
            }
        ],
    }
    schema = table.schema.with_metadata({"datapackage.json": json.dumps(datapackage).encode()})
    pq.write_table(table.cast(schema), path)
    return path


DAC_ROWS = [
    {"location": "CH", "time": 2020, "heat_demand": 6.0, "electricity_demand": 0.50,
     "temperature": 9.0, "humidity": 0.75},
    {"location": "CH", "time": 2030, "heat_demand": 5.0, "electricity_demand": 0.40,
     "temperature": 10.0, "humidity": 0.70},
    {"location": "RER", "time": 2020, "heat_demand": 6.6, "electricity_demand": 0.55,
     "temperature": 11.0, "humidity": 0.68},
    {"location": "RER", "time": 2030, "heat_demand": 5.5, "electricity_demand": 0.45,
     "temperature": 12.0, "humidity": 0.65},
]

DAC_FIELDS = [
    {"name": "location", "type": "string"},
    {"name": "time", "type": "integer", "unit": "year"},
    {"name": "heat_demand", "type": "number", "unit": "MJ",
     "iri": "https://vocab.sentier.dev/parameters/heat-demand"},
    {"name": "electricity_demand", "type": "number", "unit": "kWh",
     "iri": "https://vocab.sentier.dev/parameters/electricity-demand"},
    {"name": "temperature", "type": "number", "unit": "degC",
     "iri": "https://vocab.sentier.dev/parameters/air-temperature"},
    {"name": "humidity", "type": "number", "unit": "dimensionless",
     "iri": "https://vocab.sentier.dev/parameters/relative-humidity"},
]

parameter_file = write_parameters("dac_params.parquet", DAC_ROWS, DAC_FIELDS)
print(parameter_file)

/var/folders/l1/k90rhb0j0ns58y35ymznsd700000gn/T/tmp9a7xvksx/dac_params.parquet


## 2. Reading parameters, with fallback that says so

`ParameterSet.from_parquet` reads the rows and the descriptor together, so a
column's unit and IRI travel with its value. A lookup goes through
`params.at(location=..., time=...)` and widens until something matches: the exact
row first, then up the `LocationHierarchy`, then linear interpolation between
two bracketing years. Nothing is extrapolated past the data, and every widening
step is written into the row's `provenance`.

In [2]:
from trailrunner import LocationHierarchy, ParameterSet

hierarchy = LocationHierarchy({"CH": "RER", "FR": "RER", "RER": "GLO"})
params = ParameterSet.from_parquet(parameter_file, hierarchy=hierarchy)

row = params.at(location="CH", time=2030)
print(row["heat_demand"], row.unit_of("heat_demand"), row.iri_of("heat_demand"))
print(row.provenance)

5.0 MJ https://vocab.sentier.dev/parameters/heat-demand
{'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


France has no row of its own, and 2025 is not a year anybody wrote down. The
lookup still answers — France falls back to `RER`, and the two European rows are
interpolated — but the provenance names both substitutions: `location_used`,
`location_fallback`, `time_interpolated` and the `time_bracket` it interpolated
between. This is the whole contract: widen, but never silently.

In [3]:
fallback = params.at(location="FR", time=2025)
for column in ("heat_demand", "electricity_demand", "temperature", "humidity"):
    print(f"{column:>20}: {fallback[column]:7.3f} {fallback.unit_of(column)}")
print(fallback.provenance)

         heat_demand:   6.050 MJ
  electricity_demand:   0.500 kWh
         temperature:  11.500 degC
            humidity:   0.665 dimensionless
{'location_requested': 'FR', 'location_used': 'RER', 'location_fallback': True, 'time_requested': 2025, 'time_used': 2025, 'time_interpolated': True, 'time_bracket': (2020, 2030)}


## 3. The part that has to be code

The parquet figures assume reference air: 10 °C at 70 % relative humidity.
`ambient_penalty` scales heat and electricity when the air is something else —
colder or drier means more work per kilogram captured, warmer or wetter means
less.

The response is deliberately a plain linear one. The point of the example is
that the dependency lives in code and can be read, argued with and replaced —
not that this particular curve is right.

In [4]:
import inspect

from trailrunner.models import dac

print(inspect.getsource(dac.ambient_penalty))

def ambient_penalty(temperature: float, humidity: float) -> float:
    """Multiplier on heat and electricity demand for non-reference air.

    Colder or drier than the reference gives a value above 1.0; warmer or
    wetter gives one below. Deliberately a simple linear response: the point is
    that the dependency exists and lives in code, not that this particular
    curve is the right one.
    """
    temperature_term = TEMPERATURE_SENSITIVITY * (REFERENCE_TEMPERATURE - temperature)
    humidity_term = HUMIDITY_SENSITIVITY * (REFERENCE_HUMIDITY - humidity)
    return 1.0 + temperature_term + humidity_term



In [5]:
for temperature, humidity in [(10.0, 0.70), (0.0, 0.40), (20.0, 0.90)]:
    penalty = dac.ambient_penalty(temperature, humidity)
    print(f"{temperature:5.1f} degC, RH {humidity:.2f} -> penalty {penalty:.3f}")

 10.0 degC, RH 0.70 -> penalty 1.000
  0.0 degC, RH 0.40 -> penalty 1.190
 20.0 degC, RH 0.90 -> penalty 0.840


## 4. Answering one demand

A model is constructed with its `ParameterSet` and answers a `Demand`: a `Flow`
(what, where, when) plus an amount and a unit. `apply` receives the **full**
demanded amount, never a unit demand — a plant at ten times the scale is not ten
times the plant, and nothing downstream rescales the answer.

The `Result` has three lists:

- **production** — the demand echoed back, same flow, same unit. The Runner
  rejects a model that under-produces rather than letting the inventory shrink.
- **technosphere** — what it needs. These become demands on the queue, here heat
  and electricity at the same place and time, each in its parameter column's own
  unit (MJ and kWh).
- **biosphere** — what it exchanged with the environment. CO2 from air is
  **negative**: this process takes it out of the atmosphere.

In [6]:
from trailrunner import Demand, Flow
from trailrunner.models.dac import CO2_CAPTURED, DirectAirCapture

model = DirectAirCapture(params=params)
demand = Demand(
    flow=Flow(iri=CO2_CAPTURED, location="CH", time=2030), amount=1000.0, unit="kg"
)
result = model.apply(demand)

print("production:")
for exchange in result.production:
    print(f"  {exchange.amount:10.2f} {exchange.unit:4} {exchange.flow.iri}")
print("technosphere:")
for child in result.technosphere:
    print(f"  {child.amount:10.2f} {child.unit:4} {child.flow.iri}")
print("biosphere:")
for exchange in result.biosphere:
    print(f"  {exchange.amount:10.2f} {exchange.unit:4} {exchange.flow.iri}")
print("provenance:", result.provenance)

production:
     1000.00 kg   https://vocab.sentier.dev/products/co2-captured
technosphere:
     5000.00 MJ   https://vocab.sentier.dev/products/heat
      400.00 kWh  https://vocab.sentier.dev/products/electricity
biosphere:
    -1000.00 kg   https://vocab.sentier.dev/flows/co2-from-air
provenance: {'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


## 5. The same demand, elsewhere and later

Asking the same 1000 kg in different places and years exercises the parameter
lookup and the penalty together. Switzerland in 2030 sits exactly at reference
conditions, so its penalty is 1.0. Europe is warmer *and* slightly drier: the
warmth dominates, so the penalty lands just below 1.0 even though the underlying
heat figure is higher. France borrows Europe's row — and the `row used` column
says so.

In [7]:
header = f"{'location':>8} {'year':>6} {'degC':>6} {'RH':>5} {'penalty':>8} {'heat [MJ]':>10} {'row used':>10}"
print(header)
for location in ("CH", "FR", "RER"):
    for year in (2020, 2025, 2030):
        flow = Flow(iri=CO2_CAPTURED, location=location, time=year)
        out = model.apply(Demand(flow=flow, amount=1000.0, unit="kg"))
        heat = [d for d in out.technosphere if d.flow.iri == dac.HEAT][0]
        air = params.at(location=location, time=year)
        penalty = dac.ambient_penalty(air["temperature"], air["humidity"])
        print(
            f"{location:>8} {year:>6} {air['temperature']:>6.1f} {air['humidity']:>5.2f} "
            f"{penalty:>8.3f} {heat.amount:>10.1f} {out.provenance['location_used']:>10}"
        )

location   year   degC    RH  penalty  heat [MJ]   row used
      CH   2020    9.0  0.75    0.995     5970.0         CH
      CH   2025    9.5  0.72    0.997     5486.2         CH
      CH   2030   10.0  0.70    1.000     5000.0         CH
      FR   2020   11.0  0.68    0.996     6573.6        RER
      FR   2025   11.5  0.67    0.995     6022.8        RER
      FR   2030   12.0  0.65    0.995     5472.5        RER
     RER   2020   11.0  0.68    0.996     6573.6        RER
     RER   2025   11.5  0.67    0.995     6022.8        RER
     RER   2030   12.0  0.65    0.995     5472.5        RER


## 6. Letting the orchestrator walk outward

A `Glossary` says who produces what; the `Orchestrator` pops a demand, finds its
model, runs it, and pushes the resulting technosphere demands back onto the
queue. With only the DAC model registered, the traversal is one node deep — and
that is exactly what makes the next point visible.

The heat and electricity nobody models are reported as **cutoff leaves**, with a
reason. They are not quietly dropped and not silently zero: the unresolved list
is part of the answer, and it is the to-do list for the next model to write.
Section 7 works that list: register something that `produces` electricity and
those 400 kWh become a node of their own, with its own emissions.


In [8]:
from trailrunner import Glossary, Orchestrator

report = Orchestrator(Glossary([model])).calculate(demand)

print("inventory:")
for (flow, unit), amount in report.inventory.items():
    print(f"  {amount:10.2f} {unit:4} {flow.iri}  ({flow.location}, {flow.time})")
print("unresolved:")
for record in report.unresolved:
    print(
        f"  {record.demand.amount:10.2f} {record.demand.unit:4} "
        f"{record.demand.flow.iri}  [{record.reason}]"
    )
print("nodes:", len(report.nodes), "truncated:", report.truncated)

inventory:
    -1000.00 kg   https://vocab.sentier.dev/flows/co2-from-air  (CH, 2030)
unresolved:
     5000.00 MJ   https://vocab.sentier.dev/products/heat  [no_model_found]
      400.00 kWh  https://vocab.sentier.dev/products/electricity  [no_model_found]
nodes: 1 truncated: False


Every parameter fallback used along the way is collected per node, so the report
can be audited without re-running anything.

In [9]:
for node_id, provenance in report.provenance.items():
    print(node_id, provenance)

0 {'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


## 7. Working the to-do list: electricity

`GridElectricity` is the counterpart to the DAC model, and it earns its place as
code for the opposite reason. Nothing about it is nonlinear — what varies is
*composition*. A Swiss kilowatt hour is mostly hydro, a European one is half
fossil in 2020, and both mixes move over the decade. One emission factor per kWh
would flatten that into a number wrong in both places, so the mix is resolved per
(location, time) and split into one demand per source, each an ordinary product
with its own IRI.

Grid losses come out of the same row: what a consumer takes off the grid is less
than what was generated, so the generation demanded upstream is
`amount / (1 - grid_loss)`.

`GasPower` sits behind the gas share. It converts kilowatt hours to megajoules of
fuel at the row's efficiency — which improves with the year — and emits fossil
CO<sub>2</sub> in proportion. Wind and hydro are left unmodelled on purpose, and
so is the natural gas itself: they stay on the to-do list.

In [10]:
GRID_ROWS = [
    {"location": "CH", "time": 2020, "share_gas": 0.06, "share_wind": 0.04,
     "share_hydro": 0.90, "grid_loss": 0.070},
    {"location": "CH", "time": 2030, "share_gas": 0.02, "share_wind": 0.18,
     "share_hydro": 0.80, "grid_loss": 0.060},
    {"location": "RER", "time": 2020, "share_gas": 0.50, "share_wind": 0.30,
     "share_hydro": 0.20, "grid_loss": 0.080},
    {"location": "RER", "time": 2030, "share_gas": 0.25, "share_wind": 0.55,
     "share_hydro": 0.20, "grid_loss": 0.070},
]

GRID_FIELDS = [
    {"name": "location", "type": "string"},
    {"name": "time", "type": "integer", "unit": "year"},
    {"name": "share_gas", "type": "number", "unit": "dimensionless"},
    {"name": "share_wind", "type": "number", "unit": "dimensionless"},
    {"name": "share_hydro", "type": "number", "unit": "dimensionless"},
    {"name": "grid_loss", "type": "number", "unit": "dimensionless"},
]

# No Swiss row: the plant parameters are European, and the lookup will say so.
GAS_ROWS = [
    {"location": "RER", "time": 2020, "efficiency": 0.55, "co2_factor": 0.056},
    {"location": "RER", "time": 2030, "efficiency": 0.62, "co2_factor": 0.056},
]

GAS_FIELDS = [
    {"name": "location", "type": "string"},
    {"name": "time", "type": "integer", "unit": "year"},
    {"name": "efficiency", "type": "number", "unit": "dimensionless"},
    {"name": "co2_factor", "type": "number", "unit": "kg"},
]

grid_params = ParameterSet.from_parquet(
    write_parameters("grid_params.parquet", GRID_ROWS, GRID_FIELDS), hierarchy=hierarchy
)
gas_params = ParameterSet.from_parquet(
    write_parameters("gas_power_params.parquet", GAS_ROWS, GAS_FIELDS), hierarchy=hierarchy
)

In [11]:
from trailrunner.models.electricity import (
    CO2_FOSSIL,
    ELECTRICITY,
    ELECTRICITY_GAS,
    GasPower,
    GridElectricity,
)

grid = GridElectricity(params=grid_params)
plant = GasPower(params=gas_params)
glossary = Glossary([model, grid, plant])

full = Orchestrator(glossary).calculate(demand)


def short(iri):
    return iri.rsplit("/", 1)[-1]


print("nodes:")
for node in full.nodes:
    print(f"  {'  ' * node.depth}{node.demand.amount:9.2f} {node.demand.unit:4} {short(node.demand.flow.iri)}")
print("unresolved:")
for record in full.unresolved:
    print(
        f"  {'  ' * record.depth}{record.demand.amount:9.2f} {record.demand.unit:4} "
        f"{short(record.demand.flow.iri)}  [{record.reason}]"
    )
print("inventory:")
for (flow, unit), amount in full.inventory.items():
    print(f"  {amount:9.2f} {unit:4} {short(flow.iri)}  ({flow.location}, {flow.time})")

nodes:
    1000.00 kg   co2-captured
       400.00 kWh  electricity
           8.51 kWh  electricity-natural-gas
unresolved:
      5000.00 MJ   heat  [no_model_found]
          76.60 kWh  electricity-wind  [no_model_found]
         340.43 kWh  electricity-hydro  [no_model_found]
            49.42 MJ   natural-gas  [no_model_found]
inventory:
   -1000.00 kg   co2-from-air  (CH, 2030)
       2.77 kg   co2-fossil  (CH, 2030)


The 400 kWh cutoff is now three levels of supply chain: electricity resolves into
gas, wind and hydro generation; the gas share resolves into a plant that burns
fuel and emits. Nothing in the `Orchestrator` or the `Queue` changed to make that
happen — a model that `produces` the IRI is the whole registration.

What is left unresolved is now a *shorter, more specific* list: wind, hydro, the
natural gas, and the heat that has had no model since section 6. And the answer
has two CO<sub>2</sub> entries pulling opposite ways: the kilogram taken out of
the air by the DAC plant, and the fossil kilogram put back by the grid behind it.

Because the mix is a parameter and not a constant, the fossil side of that
balance follows place and year. Switzerland's grid is 2–6 % gas, Europe's is a
quarter to a half of it; both clean up by 2030. The gas plant has no Swiss row at
all, so its efficiency is borrowed from Europe — and `location_used` in the
node's provenance says so rather than letting the substitution pass unnoticed.

In [12]:
header = f"{'location':>8} {'year':>6} {'gas share':>10} {'fossil CO2':>11} {'net CO2':>9} {'plant row':>10}"
print(header)
for location in ("CH", "RER"):
    for year in (2020, 2030):
        report_here = Orchestrator(glossary).calculate(
            Demand(
                flow=Flow(iri=CO2_CAPTURED, location=location, time=year),
                amount=1000.0,
                unit="kg",
            )
        )
        fossil = sum(
            amount for (flow, _), amount in report_here.inventory.items()
            if flow.iri == CO2_FOSSIL
        )
        net = sum(report_here.inventory.values())
        grid_node = [n for n in report_here.nodes if n.demand.flow.iri == ELECTRICITY][0]
        plant_node = [n for n in report_here.nodes if n.demand.flow.iri == ELECTRICITY_GAS][0]
        share = report_here.provenance[grid_node.id]["shares"][ELECTRICITY_GAS]
        row_used = report_here.provenance[plant_node.id]["location_used"]
        print(
            f"{location:>8} {year:>6} {share:>10.2f} {fossil:>11.1f} {net:>9.1f} {row_used:>10}"
        )

location   year  gas share  fossil CO2   net CO2  plant row
      CH   2020       0.06        11.8    -988.2        RER
      CH   2030       0.02         2.8    -997.2        RER
     RER   2020       0.50       109.1    -890.9        RER
     RER   2030       0.25        39.1    -960.9        RER


## 8. Coverage: outside the data, the model declines

`DirectAirCapture` declares `Coverage(time_range=(2020, 2050))`. Ask it for 2015
and it is not resolved at all — but the report distinguishes *nobody models this*
(`no_model_found`) from *a registered model declined this flow*
(`coverage_excluded`), and names the model in the detail. Those are different
bugs with different fixes: write a model, or widen a coverage.

In [13]:
print(DirectAirCapture.coverage)

early = Demand(
    flow=Flow(iri=CO2_CAPTURED, location="CH", time=2015), amount=1000.0, unit="kg"
)
early_report = Orchestrator(Glossary([model])).calculate(early)

for record in early_report.unresolved:
    print(record.reason)
    print(record.detail)
print("inventory:", early_report.inventory)

Coverage(locations=None, time_range=(2020, 2050))
coverage_excluded
DirectAirCapture declares this product but its coverage does not cover location='CH' time=2015
inventory: {}


## Where this stops

The answer above is an inventory, not a score: trailrunner does no impact
characterization yet, so there is nothing to sum into a single number. Nor is
there a Brightway background — a demand nobody models stays a recorded cutoff.

Two more deliberate boundaries show up in `Report`: every visit is its own node,
never merged with an identical one elsewhere in the tree, which is what keeps
nonlinear models honest; and a cycle is truncated by the depth and node budgets
rather than solved, with `report.truncated` saying when a budget bit.

To go further from here, work the unresolved list the way section 7 did: the
README's `MyBoiler` is a ten-line model that produces the heat still cut off
above. Register it in the `Glossary` alongside the other three and that leaf
becomes a node too, with its own fuel demand and its own emissions.